# 12 · More graph ideas

Three graph ideas we have not tried, tested against the same base as the graph lab (extra trees, 200 trees, `min_samples_leaf=5`, followers + engagement + flags), 5-fold CV grouped by brand, target log10(views) on TikTok, YouTube and X. Gain = mean |error base| minus |error variant| (positive = better) with a brand-cluster bootstrap 95% CI. No tuning: every setting was fixed before the run.

- **A. Fixed aggregation features** (Rubio-Madrigal and Burkholz 2026): mean, max, std, own-minus-neighbours and 2-hop means of the engagement features over the text kNN graph. Label-free.
- **B. Cross-platform neighbours with views**: for a post, the most similar posts on the other known platforms and their views and views-minus-likes. Training-fold brands only; the post's own brand is dropped for every row.
- **C. Hierarchical shrinkage**: the shared base model plus a shrunken per-community correction (intercept, and intercept + slope on likes), learned from inner brand-grouped residuals.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
import eda_utils as eu, modeling as mo, extra_models as em, graph_lab as gl, graph_more as gm
from sklearn.model_selection import GroupKFold

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 60)

## Data

In [2]:
f, meta = em.load(ROOT)
raw = eu.load(ROOT).astype({"mention_id": str})
D = gm.build(f, meta, raw)
pd.DataFrame({eu.NAMES[p]: {"mentions": int((D["plat"] == p).sum()), "brands": len(set(D["brand"][D["plat"] == p])), "zero-reaction posts": int(gm.zero_mask(D["X"])[D["plat"] == p].sum())} for p in gm.PLATS})

,TikTok,YouTube,X (Twitter)
mentions,2491,2441,1803
brands,266,273,252
zero-reaction posts,90,665,991


## In-platform variants (A, B, A + B, C)

In [3]:
Ps, preds, tabs = {}, {}, []
for p in gm.PLATS:
    Ps[p], preds[p] = gm.run_platform(D, p); tabs.append(gm.table(Ps[p], preds[p], p))
res = pd.concat(tabs, ignore_index=True)
for grp in ["all posts", "zero reactions"]:
    r = res[res.group == grp]
    print(grp); display(r.pivot(index="variant", columns="platform", values="MAE").reindex(r.variant.unique()).round(3))

all posts


platform,TikTok,X (Twitter),YouTube
variant,,,
base,0.277,0.481,0.455
A: fixed aggregation (label-free),0.281,0.475,0.454
B: cross-platform neighbours with views,0.277,0.480,0.446
A + B,0.280,0.476,0.449
C: community intercept (shrunk),0.278,0.479,0.457
C: community intercept + reactions slope (shrunk),0.279,0.481,0.453


zero reactions


platform,TikTok,X (Twitter),YouTube
variant,,,
base,0.534,0.510,0.655
A: fixed aggregation (label-free),0.543,0.501,0.660
B: cross-platform neighbours with views,0.538,0.503,0.636
A + B,0.539,0.499,0.652
C: community intercept (shrunk),0.538,0.509,0.656
C: community intercept + reactions slope (shrunk),0.537,0.511,0.651


Paired gain over the base, with 95% CI (low, high).

In [4]:
g = res.assign(txt=lambda d: d.gain.round(3).astype(str) + " [" + d["gain low"].round(3).astype(str) + ", " + d["gain high"].round(3).astype(str) + "]")
for grp in ["all posts", "zero reactions"]:
    r = g[(g.group == grp) & (g.variant != "base")]; print(grp, "(n =", dict(r.groupby("platform").n.first()), ")")
    display(r.pivot(index="variant", columns="platform", values="txt").reindex(r.variant.unique()))

all posts (n = {'TikTok': np.int64(2491), 'X (Twitter)': np.int64(1803), 'YouTube': np.int64(2441)} )


platform,TikTok,X (Twitter),YouTube
variant,,,
A: fixed aggregation (label-free),"-0.003 [-0.006, -0.0]","0.006 [-0.001, 0.013]","0.001 [-0.005, 0.008]"
B: cross-platform neighbours with views,"0.0 [-0.001, 0.002]","0.002 [-0.002, 0.006]","0.009 [0.005, 0.013]"
A + B,"-0.003 [-0.006, 0.0]","0.006 [-0.001, 0.013]","0.006 [-0.0, 0.012]"
C: community intercept (shrunk),"-0.001 [-0.004, 0.002]","0.002 [-0.004, 0.008]","-0.002 [-0.005, 0.001]"
C: community intercept + reactions slope (shrunk),"-0.002 [-0.006, 0.002]","0.001 [-0.005, 0.008]","0.002 [-0.004, 0.007]"


zero reactions (n = {'TikTok': np.int64(90), 'X (Twitter)': np.int64(991), 'YouTube': np.int64(665)} )


platform,TikTok,X (Twitter),YouTube
variant,,,
A: fixed aggregation (label-free),"-0.01 [-0.046, 0.026]","0.009 [-0.001, 0.021]","-0.004 [-0.024, 0.016]"
B: cross-platform neighbours with views,"-0.004 [-0.033, 0.02]","0.007 [0.001, 0.014]","0.019 [0.006, 0.032]"
A + B,"-0.005 [-0.044, 0.034]","0.011 [-0.0, 0.023]","0.003 [-0.016, 0.023]"
C: community intercept (shrunk),"-0.004 [-0.016, 0.008]","0.001 [-0.006, 0.009]","-0.001 [-0.007, 0.006]"
C: community intercept + reactions slope (shrunk),"-0.003 [-0.026, 0.02]","-0.001 [-0.008, 0.008]","0.004 [-0.008, 0.017]"


## C · do reaction slopes differ between communities?
Permutation test on the out-of-fold residual of the base: spread (SD) of shrunken per-community slopes against shuffled community labels.

In [5]:
het = {eu.NAMES[p]: gm.slope_heterogeneity(Ps[p], Ps[p]["comm"], pd.Series(preds[p]["base"])) for p in gm.PLATS}
pd.DataFrame(het).T.round(4)

,communities,slope SD observed,slope SD null mean,p (permutation)
TikTok,28.0,0.0551,0.0389,0.0033
YouTube,24.0,0.1020,0.0510,0.0033
X (Twitter),20.0,0.0667,0.0531,0.0664


## B as a bridge · leave-one-platform-out
Train on the other two known platforms with the test fold's brands removed; test on the left-out platform. Ridge on likes + comments is the reference with known level shift.

In [6]:
bt = gm.bridge_table(D)
b = bt[(bt.group == "all posts")]
display(b.pivot_table(index=["model", "variant"], columns="platform", values=["MAE", "bias"]).round(3))
z = bt[(bt.group == "zero reactions")]
display(z.assign(txt=lambda d: d.gain.round(3).astype(str) + " [" + d["gain low"].round(3).astype(str) + ", " + d["gain high"].round(3).astype(str) + "]").query("variant != 'without bridge'").pivot_table(index="model", columns="platform", values="txt", aggfunc="first"))

MAE                       bias                    
platform                               TikTok X (Twitter) YouTube TikTok X (Twitter) YouTube
model       variant                                                                         
extra trees with cross-platform bridge  0.361       0.673   0.631  0.027      -0.436   0.123
            without bridge              0.362       0.663   0.614  0.029      -0.438   0.127
ridge       with cross-platform bridge  0.309       0.650   0.608  0.085      -0.401   0.209
            without bridge              0.313       0.649   0.612  0.029      -0.402   0.208

platform,TikTok,X (Twitter),YouTube
model,,,
extra trees,"-0.004 [-0.034, 0.022]","-0.023 [-0.034, -0.012]","-0.031 [-0.04, -0.022]"
ridge,"0.018 [0.002, 0.035]","-0.001 [-0.003, 0.001]","0.022 [0.015, 0.031]"


## Leakage checks

In [7]:
P = Ps["youtube"]; others = ["tiktok", "twitter"]; chk = []
for tr, te in GroupKFold(5).split(P["X"], P["y"], P["g"]):
    test_b = set(P["g"].iloc[te]); allowed = ~pd.Series(D["brand"]).isin(test_b).values
    pool = np.where(np.isin(D["plat"], others) & allowed)[0]
    chk.append({"test brands in the pool": len(set(D["brand"][pool]) & test_b), "own-platform rows in the pool": int((D["plat"][pool] == "youtube").sum()), "train brands overlapping test brands": len(set(P["g"].iloc[tr]) & test_b)})
display(pd.DataFrame(chk))
# placebo: shuffle the views of the pool; a real signal must disappear
rng = np.random.default_rng(42); y2 = D["y"].copy(); y2[D["plat"] != "youtube"] = rng.permutation(y2[D["plat"] != "youtube"]); D2 = dict(D, y=y2)
pl = em.cv_predict(gm.make_et, P["X"], P["y"], P["g"], gm.xp_extra(D2, P, "youtube"))
placebo = dict(zip(["gain", "gain low", "gain high"], gm.ni.paired_diff(P["y"].values, preds["youtube"]["base"], pl, P["g"].values)))
placebo

,test brands in the pool,own-platform rows in the pool,train brands overlapping test brands
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0


{'gain': np.float64(0.00313958524181258),
 'gain low': np.float64(-0.000566062114282995),
 'gain high': np.float64(0.006861909501908435)}

## Figures and numbers

In [8]:
fd = ROOT / "figures"
for lang in ["en", "pl"]:
    gm.plot_gains(res, "all posts", fd / f"graphmore_gains_{lang}.png", lang); gm.plot_gains(res, "zero reactions", fd / f"graphmore_zero_{lang}.png", lang); gm.plot_bridge(bt, fd / f"graphmore_bridge_{lang}.png", lang)
out = {"in_platform": res.to_dict("records"), "slope_heterogeneity": het, "bridge": bt.to_dict("records"), "placebo_B_youtube": placebo, "leakage_checks": chk}
(ROOT / "data" / "processed" / "graph_more_numbers.json").write_text(json.dumps(out, indent=1, default=float)); "saved"

'saved'

## Takeaways

- **Nothing is a large gain.** Largest all-posts gain with a CI above zero: B on YouTube, +0.009 log10 (CI 0.005 to 0.013), on a base MAE of 0.455. Same size class as the earlier graph tries (at most about 0.007).
- **A (fixed aggregation, label-free)**: no reliable gain; TikTok slightly worse (-0.003, CI -0.006 to 0.000). The graph adds no more than the plain neighbour means tried before.
- **B (cross-platform neighbours with views)**: small gain on YouTube; none on TikTok and X for all posts. For posts with zero reactions it helps YouTube (+0.019, CI 0.006 to 0.032) and X (+0.007, CI 0.001 to 0.014). The placebo with shuffled views gives +0.003 on YouTube, so part of the gain is label-free content of neighbours, not their views.
- **C (hierarchical shrinkage)**: no gain. Slopes of reactions differ between communities more than by chance on TikTok and YouTube (permutation p about 0.003), but the shrunken correction does not turn that into lower error.
- **Bridge**: the level shift does not shrink. Ridge bias stays +0.03 / +0.21 / -0.40 (TikTok moves to +0.085). Ridge MAE changes by at most 0.004. With trees the bridge hurts YouTube and X. For zero-reaction posts the ridge bridge gains on TikTok (+0.018, n=90) and YouTube (+0.022).
- Many comparisons (5 variants x 3 platforms x 2 groups); isolated CIs above zero should be read as hints, not confirmed gains.
- Skipped: GCN / GraphSAGE (no torch in the environment, and the fixed-aggregation result of Rubio-Madrigal and Burkholz suggests learned aggregation adds little when mean aggregation is already tried), post-author-brand-hashtag metapaths (author, hashtag and brand graphs already tested; brand-level cross-platform labels are unavailable for a held-out brand).